In [1]:
import pyspark

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.master( "local[3]" ) \
    .appName('Spark') \
    .getOrCreate()

In [22]:
spark = SparkSession.builder \
    .appName("myApp") \
    .config("spark.sql.shuffle.partitions", 10) \
    .getOrCreate()

#config("spark.sql.adaptive.enabled", "true") \
#.config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
#.config("spark.sql.adaptive.skewJoin.enabled", "true") \
#.set("spark.executor.memory", "4g")

In [4]:
print("App Name:", spark.sparkContext.appName)
#print("Master:" spark.sparkContext.master)

App Name: Spark


In [6]:
sc = spark.sparkContext

In [7]:
# Set log level to show more detailed information (e.g., INFO, DEBUG)
sc.setLogLevel("INFO")

# Or set to a higher level to reduce noise (e.g., WARN, ERROR, FATAL)
# sc.setLogLevel("WARN")

In [9]:
spark.conf.set("spark.sql.shuffle.partitions", 11)

In [20]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [ ]:
#spark.executor.memory 4g
#spark.driver.memory 2g

In [10]:
# No Schema RDD
rdd = sc.parallelize([1,2,3,4])
print(rdd.map(lambda x:x * 2).collect())

[2, 4, 6, 8]


In [11]:
rdd.getNumPartitions()

3

In [12]:
# Schema tied DF

df = spark.createDataFrame([(1,"A"),(2,"B"),(3,"C")], ["id", "name"])

# Transformations
filtered_df = df.filter('id > 1') #df.id df['id']
new_col_df = df.withColumn('id squared', df.id * df.id)

# Execution on Actions
filtered_df.show()
new_col_df.show()

+---+----+
| id|name|
+---+----+
|  2|   B|
|  3|   C|
+---+----+

+---+----+----------+
| id|name|id squared|
+---+----+----------+
|  1|   A|         1|
|  2|   B|         4|
|  3|   C|         9|
+---+----+----------+



In [13]:
df = spark.range(1, 6)
df.show()

# Actions
row_count = df.count()
all_rows = df.collect()

print("Row Count:", row_count)
print("Rows:", all_rows)

+---+
| id|
+---+
|  1|
|  2|
|  3|
|  4|
|  5|
+---+

Row Count: 5
Rows: [Row(id=1), Row(id=2), Row(id=3), Row(id=4), Row(id=5)]


In [15]:
df = spark.range(1,10)

# Lazy Transformations
df2 = df.filter('id > 5').withColumn('double_id', df.id * 2) #df.id

# Execution happens here
df2.show()

+---+---------+
| id|double_id|
+---+---------+
|  6|       12|
|  7|       14|
|  8|       16|
|  9|       18|
+---+---------+



In [16]:
df = spark.range(1, 100)
optimized = df.filter('id > 50').select('id')
optimized.explain(True)

== Parsed Logical Plan ==
'Project ['id]
+- Filter (id#63L > cast(50 as bigint))
   +- Range (1, 100, step=1, splits=Some(3))

== Analyzed Logical Plan ==
id: bigint
Project [id#63L]
+- Filter (id#63L > cast(50 as bigint))
   +- Range (1, 100, step=1, splits=Some(3))

== Optimized Logical Plan ==
Filter (id#63L > 50)
+- Range (1, 100, step=1, splits=Some(3))

== Physical Plan ==
*(1) Filter (id#63L > 50)
+- *(1) Range (1, 100, step=1, splits=3)



In [17]:
df = spark.range(1, 100)
optimized = df.filter('id > 50').select('id')
optimized.explain(mode='extended')

== Parsed Logical Plan ==
'Project ['id]
+- Filter (id#68L > cast(50 as bigint))
   +- Range (1, 100, step=1, splits=Some(3))

== Analyzed Logical Plan ==
id: bigint
Project [id#68L]
+- Filter (id#68L > cast(50 as bigint))
   +- Range (1, 100, step=1, splits=Some(3))

== Optimized Logical Plan ==
Filter (id#68L > 50)
+- Range (1, 100, step=1, splits=Some(3))

== Physical Plan ==
*(1) Filter (id#68L > 50)
+- *(1) Range (1, 100, step=1, splits=3)



In [18]:
df = spark.range(1, 1000000)
df_filtered = df.filter(df.id % 2 == 0)
df_filtered.explain(True)

== Parsed Logical Plan ==
'Filter ((id#73L % 2) = 0)
+- Range (1, 1000000, step=1, splits=Some(3))

== Analyzed Logical Plan ==
id: bigint
Filter ((id#73L % cast(2 as bigint)) = cast(0 as bigint))
+- Range (1, 1000000, step=1, splits=Some(3))

== Optimized Logical Plan ==
Filter ((id#73L % 2) = 0)
+- Range (1, 1000000, step=1, splits=Some(3))

== Physical Plan ==
*(1) Filter ((id#73L % 2) = 0)
+- *(1) Range (1, 1000000, step=1, splits=3)



In [19]:
df = spark.range(1, 10)
result = df.filter('id % 2 == 0').groupBy().count()
result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[count(1)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=143]
      +- HashAggregate(keys=[], functions=[partial_count(1)])
         +- Project
            +- Filter ((id#77L % 2) = 0)
               +- Range (1, 10, step=1, splits=3)




In [21]:
df = spark.range(1, 10)

# Narrow
narrow_df = df.filter(df.id < 5)
narrow_df.explain()

#Wide
wide_df = df.groupBy((df.id % 2).alias("parity")).count()
wide_df.explain()

narrow_df.show()
wide_df.show()

== Physical Plan ==
*(1) Filter (id#87L < 5)
+- *(1) Range (1, 10, step=1, splits=3)


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[_groupingexpression#97L], functions=[count(1)])
   +- Exchange hashpartitioning(_groupingexpression#97L, 11), ENSURE_REQUIREMENTS, [plan_id=168]
      +- HashAggregate(keys=[_groupingexpression#97L], functions=[partial_count(1)])
         +- Project [(id#87L % 2) AS _groupingexpression#97L]
            +- Range (1, 10, step=1, splits=3)


+---+
| id|
+---+
|  1|
|  2|
|  3|
|  4|
+---+

+------+-----+
|parity|count|
+------+-----+
|     1|    5|
|     0|    4|
+------+-----+



In [14]:
df = spark.range(1, 20).repartition(4)
print("Partitions:", df.rdd.getNumPartitions()) #converting to rdd

Partitions: 4


In [22]:
df = spark.range(1, 10)
shuffled = df.groupBy((df.id % 3).alias("grp")).count()
shuffled.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[_groupingexpression#123L], functions=[count(1)])
   +- Exchange hashpartitioning(_groupingexpression#123L, 11), ENSURE_REQUIREMENTS, [plan_id=234]
      +- HashAggregate(keys=[_groupingexpression#123L], functions=[partial_count(1)])
         +- Project [(id#115L % 3) AS _groupingexpression#123L]
            +- Range (1, 10, step=1, splits=3)




In [23]:
df = spark.range(1, 100)

# Cache
df_cache = df.cache()

# Persist
from pyspark import StorageLevel
df_persisted = df.persist(StorageLevel.MEMORY_AND_DISK)

In [24]:
from pyspark.sql.functions import broadcast

large_df = spark.range(1, 1000000).withColumnRenamed("id", "key")
small_df = spark.createDataFrame([(1,),(2,),(3,)], ["key"])

large_df.join(small_df, 'key').explain()

joined = large_df.join(broadcast(small_df), "key")
joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [key#135L]
   +- BroadcastHashJoin [key#135L], [key#137L], Inner, BuildLeft, false
      :- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=264]
      :  +- Project [id#133L AS key#135L]
      :     +- Range (1, 1000000, step=1, splits=3)
      +- Filter isnotnull(key#137L)
         +- Scan ExistingRDD[key#137L]


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [key#135L]
   +- BroadcastHashJoin [key#135L], [key#137L], Inner, BuildRight, false
      :- Project [id#133L AS key#135L]
      :  +- Range (1, 1000000, step=1, splits=3)
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=291]
         +- Filter isnotnull(key#137L)
            +- Scan ExistingRDD[key#137L]




In [25]:
joined.show(7)

+---+
|key|
+---+
|  1|
|  2|
|  3|
+---+



In [32]:
nobel_ds = "/Users/abhinavmohanty/Documents/Python/DataCamp/Data Scientist/Projects/A Visual History of Nobel Prize Winners/datasets/nobel.csv"
df_inferred = spark.read.csv(nobel_ds, header=True, inferSchema=True)
df_inferred.printSchema()

root
 |-- year: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- prize: string (nullable = true)
 |-- motivation: string (nullable = true)
 |-- prize_share: string (nullable = true)
 |-- laureate_id: string (nullable = true)
 |-- laureate_type: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- birth_city: string (nullable = true)
 |-- birth_country: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- organization_name: string (nullable = true)
 |-- organization_city: string (nullable = true)
 |-- organization_country: string (nullable = true)
 |-- death_date: string (nullable = true)
 |-- death_city: string (nullable = true)
 |-- death_country: string (nullable = true)

+----------+-----+
|  category|count|
+----------+-----+
| Chemistry|  175|
|  Medicine|  211|
|   Physics|  204|
|Literature|  113|
| Economics|   78|
|     Peace|  130|
+----------+-----+



In [33]:
df_inferred.groupBy("category").count().explain()

== Physical Plan ==
*(2) HashAggregate(keys=[category#269], functions=[count(1)])
+- Exchange hashpartitioning(category#269, 200), ENSURE_REQUIREMENTS, [id=#425]
   +- *(1) HashAggregate(keys=[category#269], functions=[partial_count(1)])
      +- FileScan csv [category#269] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex[file:/Users/abhinavmohanty/Documents/Python/DataCamp/Data Scientist/Projects/A ..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<category:string>




In [26]:
# DataFrome Schema Aware
df = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])
df.show()

rdd = df.rdd.map(lambda row: (row[0], row[1].upper()))
print(rdd.collect())

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+

[(1, 'ALICE'), (2, 'BOB')]


In [27]:
# create DataFrame
data = [("Alice", 23), ("Bob", 29), ("Charlie", 37)]
df = spark.createDataFrame(data, ["name", "age"])

# Register DataFrame as temporary SQL view
df.createOrReplaceTempView("people")

# Run SQL query
sql_result = spark.sql("Select name, age from people where age > 25")

sql_result.show()

+-------+---+
|   name|age|
+-------+---+
|    Bob| 29|
|Charlie| 37|
+-------+---+



In [28]:
data = [("Alice", 23), ("Bob", 29)]
df = spark.createDataFrame(data, ["name", "age"])

result = df.filter("age > 20")
result.show()

+-----+---+
| name|age|
+-----+---+
|Alice| 23|
|  Bob| 29|
+-----+---+



In [29]:
from pyspark.sql.functions import udf

In [30]:
# Sample sales DataFrame
data = [("A", 100), ("B", 200), ("C", 150), ("A", 120), ("C", 250), ("B", 160), ("A", 170)]
columns = ["product_id", "amount"]

sales_df = spark.createDataFrame(data, columns)

# Lookup dictionary for Categories
product_lookup = {"A": "Electronics", "B": "Clothing", "C": "Books"}

# Broadcast the lookup
broadcast_lookup = spark.sparkContext.broadcast(product_lookup)

@udf
def get_category(product_id):
    return broadcast_lookup.value.get(product_id, "Unknown")

# Add category column to the DataFrame
sales_df = sales_df.withColumn("category", get_category("product_id"))
sales_df.explain()
sales_df.show()

== Physical Plan ==
*(2) Project [product_id#187, amount#188L, pythonUDF0#196 AS category#192]
+- BatchEvalPython [get_category(product_id#187)#191], [pythonUDF0#196]
   +- *(1) Scan ExistingRDD[product_id#187,amount#188L]


+----------+------+-----------+
|product_id|amount|   category|
+----------+------+-----------+
|         A|   100|Electronics|
|         B|   200|   Clothing|
|         C|   150|      Books|
|         A|   120|Electronics|
|         C|   250|      Books|
|         B|   160|   Clothing|
|         A|   170|Electronics|
+----------+------+-----------+



In [31]:
data = [("US", 100), ("IN", 200), ("US", 50), ("IN", 75)]
rdd = spark.sparkContext.parallelize(data)

# Ex reduceByKey to sum sales
result = rdd.reduceByKey(lambda a, b: a + b).collect()
print(result)

[('US', 150), ('IN', 275)]


In [32]:
# Sample k-v RDD
data = [("apple", 1), ("banana", 3), ("apple", 5), ("banana", 7), ("orange", 11)]
rdd = spark.sparkContext.parallelize(data) ## parallelize method of sparkContext

# groupByKey
grouped = rdd.groupByKey()

# Collect Result
for k, v in grouped.collect():
    print(k, list(v))

orange [11]
banana [3, 7]
apple [1, 5]


In [33]:
# Sum of values for each fruit
reduced = rdd.reduceByKey( lambda x, y: x + y)

print(reduced.collect())

[('orange', 11), ('banana', 10), ('apple', 6)]


In [29]:
df = spark.range(0, 20).repartition(6) # create 6 partitions
print("Partitions Before:", df.rdd.getNumPartitions())

df2 = df.coalesce(2) # reduce to 2 partitions without shuffle
print("Partitions After:", df2.rdd.getNumPartitions())

df3 = df2.repartition(10) # increase to 10 partitions with shuffle
print("Partitions after repartitions:", df3.rdd.getNumPartitions())

Partitions Before: 6
Partitions After: 2
Partitions after repartitions: 10


In [5]:
data = [(1, 'Alice', 'DE'), (2, 'Bob', 'SE'), (3, 'Charlie', 'DA'), (4, 'David', 'DA')]
schema = 'id int, name string, designation string'
df = spark.createDataFrame(data, schema)
display(df)

DataFrame[id: int, name: string, designation: string]

In [6]:
df.show()

+---+-------+-----------+
| id|   name|designation|
+---+-------+-----------+
|  1|  Alice|         DE|
|  2|    Bob|         SE|
|  3|Charlie|         DA|
|  4|  David|         DA|
+---+-------+-----------+



In [5]:
df.write.format('delta').mode('overwrite').saveAsTable('employee_detail')

Py4JJavaError: An error occurred while calling o50.saveAsTable.
: java.lang.ClassNotFoundException: Failed to find data source: delta. Please find packages at http://spark.apache.org/third-party-projects.html
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:689)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSourceV2(DataSource.scala:743)
	at org.apache.spark.sql.DataFrameWriter.lookupV2Provider(DataFrameWriter.scala:993)
	at org.apache.spark.sql.DataFrameWriter.saveAsTable(DataFrameWriter.scala:615)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:566)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4

In [9]:
df.write.format('delta').mode('append').option('overwriteSchema', 'true') \
    .save('/Users/abhinavmohanty/Documents/Python/test_delta')

Py4JJavaError: An error occurred while calling o54.save.
: org.apache.spark.SparkClassNotFoundException: [DATA_SOURCE_NOT_FOUND] Failed to find the data source: delta. Please find packages at `https://spark.apache.org/third-party-projects.html`.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.dataSourceNotFoundError(QueryExecutionErrors.scala:725)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:647)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSourceV2(DataSource.scala:697)
	at org.apache.spark.sql.DataFrameWriter.lookupV2Provider(DataFrameWriter.scala:873)
	at org.apache.spark.sql.DataFrameWriter.saveInternal(DataFrameWriter.scala:260)
	at org.apache.spark.sql.DataFrameWriter.save(DataFrameWriter.scala:243)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at java.base/jdk.inter

----------------------------------------
Exception happened during processing of request from ('127.0.0.1', 52345)
Traceback (most recent call last):
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/socketserver.py", line 720, in __init__
    self.handle()
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/site-packages/pyspark/accumulators.py", line 267, in 

In [8]:
# df.write.format('parquet').mode('overwrite').option('overwriteSchema', 'true') \
    #.save('/Users/abhinavmohanty/Documents/Python/test_parquet')

In [32]:
pip install delta-spark

     |████████████████████████████████| 317.4 MB 4.0 MB/s eta 0:00:01     |██████▋                         | 65.7 MB 3.4 MB/s eta 0:01:14
     |████████████████████████████████| 200 kB 5.7 MB/s eta 0:00:01
  Created wheel for pyspark: filename=pyspark-3.5.7-py2.py3-none-any.whl size=317907736 sha256=ce2ac1651f13001084fde3244784437c2d09d2af81f973cc663bb8790e0f831b
  Stored in directory: /Users/abhinavmohanty/Library/Caches/pip/wheels/90/7b/3a/f4e0b629a2ec0ac23c6ea92ee30d7fb269def367fd9f70bcf9
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9
    Uninstalling py4j-0.10.9:
      Successfully uninstalled py4j-0.10.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 3.1.1
    Uninstalling pyspark-3.1.1:
      Successfully uninstalled pyspark-3.1.1
Note: you may need to restart the kernel to use updated packages.


In [8]:
from delta import *

In [12]:
import datetime

In [4]:
from pyspark.sql.functions import date_format

In [8]:
from pyspark.sql import functions as F

In [15]:
from pyspark.sql.functions import col, lit, concat, left, split, right

In [10]:
print(date_format("2026-09-26", "yyyy-MM"))

TypeError: 'Column' object is not callable

In [17]:
print(sf.current_date())

Column<'current_date()'>


AttributeError: 'NoneType' object has no attribute 'show'

In [19]:
spark.range(1).select(sf.current_date()).show()

+--------------+
|current_date()|
+--------------+
|    2026-04-22|
+--------------+



In [15]:
df = spark.createDataFrame([('2015-04-08',), ('2024-10-31',)], ['dt'])
df.select("*", sf.typeof('dt'), sf.date_format('dt', 'yyyy-MM')).show() #MM/dd/yyyy

+----------+----------+------------------------+
|        dt|typeof(dt)|date_format(dt, yyyy-MM)|
+----------+----------+------------------------+
|2015-04-08|    string|                 2015-04|
|2024-10-31|    string|                 2024-10|
+----------+----------+------------------------+



In [11]:
df = spark.createDataFrame([('2015-04-08 13:08:15',), ('2024-10-31 10:09:16',)], ['ts'])
df.select("*", sf.typeof('ts'), sf.date_format('ts', 'yy=MM=dd HH=mm=ss')).show()

+-------------------+----------+----------------------------------+
|                 ts|typeof(ts)|date_format(ts, yy=MM=dd HH=mm=ss)|
+-------------------+----------+----------------------------------+
|2015-04-08 13:08:15|    string|                 15=04=08 13=08=15|
|2024-10-31 10:09:16|    string|                 24=10=31 10=09=16|
+-------------------+----------+----------------------------------+



In [13]:
df = spark.createDataFrame([
    (datetime.date(2015, 4, 8),),
    (datetime.date(2024, 10, 31),)], ['dt'])
df.select("*", sf.typeof('dt'), sf.date_format('dt', 'yy--MM--dd')).show()

+----------+----------+---------------------------+
|        dt|typeof(dt)|date_format(dt, yy--MM--dd)|
+----------+----------+---------------------------+
|2015-04-08|      date|                 15--04--08|
|2024-10-31|      date|                 24--10--31|
+----------+----------+---------------------------+



In [14]:
df = spark.createDataFrame([
    (datetime.datetime(2015, 4, 8, 13, 8, 15),),
    (datetime.datetime(2024, 10, 31, 10, 9, 16),)], ['ts'])
df.select("*", sf.typeof('ts'), sf.date_format('ts', 'yy=MM=dd HH=mm=ss')).show()

+-------------------+----------+----------------------------------+
|                 ts|typeof(ts)|date_format(ts, yy=MM=dd HH=mm=ss)|
+-------------------+----------+----------------------------------+
|2015-04-08 13:08:15| timestamp|                 15=04=08 13=08=15|
|2024-10-31 10:09:16| timestamp|                 24=10=31 10=09=16|
+-------------------+----------+----------------------------------+



In [14]:
spark.range(4).withColumn('id', F.when(F.col('id') == 1, F.lit(True)).otherwise(F.lit(False))) \ # test
    .show()
    #.drop('id') \
    

+-----+
|   id|
+-----+
|false|
| true|
|false|
|false|
+-----+



In [17]:
partition_cols = ['urid', 'srid']
print(', '.join(partition_cols) if partition_cols else "")

urid, srid


In [22]:
merge_keys = ['urid', 'srid']
print(f"target.{k} = source.{k}" for k in merge_keys)
print( " AND ".join( [f"target.{k} = source.{k}" for k in merge_keys] ) )

<generator object <genexpr> at 0x7fc578669200>
target.urid = source.urid AND target.srid = source.srid


In [35]:
BRONZE_SCHEMA      = "bronze"
SILVER_SCHEMA      = "silver"
GOLD_SCHEMA        = "gold"

schema_tags = {
        BRONZE_SCHEMA: "layer=bronze,sensitivity=raw,pii=true",
        SILVER_SCHEMA: "layer=silver,sensitivity=conformd,pii=true",
        GOLD_SCHEMA:   "layer=gold,sensitivity=aggregated,pii=partial",
}
    
for schema, tag_str in schema_tags.items():
        tags_kv = ",\n            ".join(
            f"'{k}' = '{v}'" for pair in tag_str.split(",")
            for k, v in [pair.split("=")]
        )
        print(tags_kv)

'layer' = 'bronze',
            'sensitivity' = 'raw',
            'pii' = 'true'
'layer' = 'silver',
            'sensitivity' = 'conformd',
            'pii' = 'true'
'layer' = 'gold',
            'sensitivity' = 'aggregated',
            'pii' = 'partial'


In [40]:

    table_tag_map = {
        # Bronze
        f"{BRONZE_SCHEMA}.subscription_raw": {"entity": "subscription", "pii": "false", "criticality": "high"},
        f"{BRONZE_SCHEMA}.user_raw":         {"entity": "user",         "pii": "true",  "criticality": "critical"},
        f"{BRONZE_SCHEMA}.catalog_raw":      {"entity": "catalog",      "pii": "false", "criticality": "medium"},
        f"{BRONZE_SCHEMA}.order_raw":        {"entity": "order",        "pii": "partial","criticality": "high"},
        f"{BRONZE_SCHEMA}.device_raw":       {"entity": "device",       "pii": "false", "criticality": "medium"},
        f"{BRONZE_SCHEMA}.payment_raw":      {"entity": "payment",      "pii": "true",  "criticality": "critical"},
        # Silver
        f"{SILVER_SCHEMA}.subscription":     {"entity": "subscription", "pii": "false", "criticality": "high"},
        f"{SILVER_SCHEMA}.user":             {"entity": "user",         "pii": "true",  "criticality": "critical"},
        f"{SILVER_SCHEMA}.catalog":          {"entity": "catalog",      "pii": "false", "criticality": "medium"},
        f"{SILVER_SCHEMA}.order":            {"entity": "order",        "pii": "partial","criticality": "high"},
        f"{SILVER_SCHEMA}.device":           {"entity": "device",       "pii": "false", "criticality": "medium"},
        f"{SILVER_SCHEMA}.payment":          {"entity": "payment",      "pii": "true",  "criticality": "critical"},
        f"{SILVER_SCHEMA}.subscription_enriched": {"entity": "subscription_enriched", "pii": "true", "criticality": "high"},
        # Gold
        f"{GOLD_SCHEMA}.agg_subscription_metrics_daily":   {"entity": "agg_sub_daily",   "pii": "false", "criticality": "medium"},
        f"{GOLD_SCHEMA}.agg_subscription_metrics_monthly": {"entity": "agg_sub_monthly", "pii": "false", "criticality": "medium"},
        f"{GOLD_SCHEMA}.agg_revenue_by_plan":               {"entity": "agg_revenue",     "pii": "false", "criticality": "high"},
        f"{GOLD_SCHEMA}.agg_user_cohort_analysis":          {"entity": "agg_cohort",      "pii": "false", "criticality": "medium"},
        f"{GOLD_SCHEMA}.agg_device_distribution":           {"entity": "agg_device",      "pii": "false", "criticality": "low"},
        f"{GOLD_SCHEMA}.agg_payment_performance":           {"entity": "agg_payment",     "pii": "false", "criticality": "high"},
        f"{GOLD_SCHEMA}.dim_subscription_status_snapshot":  {"entity": "dim_sub_status",  "pii": "partial","criticality": "high"},
        f"{GOLD_SCHEMA}.fact_subscription_lifecycle":       {"entity": "fact_lifecycle",  "pii": "false", "criticality": "high"},
    }

    for table_path, tags in table_tag_map.items():
        tags_sql = ",\n            ".join(f"'{k}' = '{v}'" for k, v in tags.items())
        print(f"{table_path} : {tags_sql}")

bronze.subscription_raw : 'entity' = 'subscription',
            'pii' = 'false',
            'criticality' = 'high'
bronze.user_raw : 'entity' = 'user',
            'pii' = 'true',
            'criticality' = 'critical'
bronze.catalog_raw : 'entity' = 'catalog',
            'pii' = 'false',
            'criticality' = 'medium'
bronze.order_raw : 'entity' = 'order',
            'pii' = 'partial',
            'criticality' = 'high'
bronze.device_raw : 'entity' = 'device',
            'pii' = 'false',
            'criticality' = 'medium'
bronze.payment_raw : 'entity' = 'payment',
            'pii' = 'true',
            'criticality' = 'critical'
silver.subscription : 'entity' = 'subscription',
            'pii' = 'false',
            'criticality' = 'high'
silver.user : 'entity' = 'user',
            'pii' = 'true',
            'criticality' = 'critical'
silver.catalog : 'entity' = 'catalog',
            'pii' = 'false',
            'criticality' = 'medium'
silver.order : 'entity' = 'ord

In [44]:
pii_column_tags = {
        f"{SILVER_SCHEMA}.user": {
            "email":          {"pii_type": "email",         "gdpr_subject": "true"},
            "full_name":      {"pii_type": "name",          "gdpr_subject": "true"},
            "phone_number":   {"pii_type": "phone",         "gdpr_subject": "true"},
            "date_of_birth":  {"pii_type": "dob",           "gdpr_subject": "true"},
            "address":        {"pii_type": "address",       "gdpr_subject": "true"},
        },
        f"{SILVER_SCHEMA}.payment": {
            "card_number":    {"pii_type": "financial_pci", "pci_dss": "true"},
            "bank_account":   {"pii_type": "financial_pci", "pci_dss": "true"},
            "billing_address":{"pii_type": "address",       "gdpr_subject": "true"},
        },
        f"{SILVER_SCHEMA}.order": {
            "shipping_address":{"pii_type": "address",      "gdpr_subject": "true"},
        },
    }

for table_path, cols in pii_column_tags.items():
    for col_name, col_tags in cols.items():
        col_tags_sql = ", ".join(f"'{k}' = '{v}'" for k, v in col_tags.items())
        print(f"{table_path} : {col_tags_sql}") 

silver.user : 'pii_type' = 'email', 'gdpr_subject' = 'true'
silver.user : 'pii_type' = 'name', 'gdpr_subject' = 'true'
silver.user : 'pii_type' = 'phone', 'gdpr_subject' = 'true'
silver.user : 'pii_type' = 'dob', 'gdpr_subject' = 'true'
silver.user : 'pii_type' = 'address', 'gdpr_subject' = 'true'
silver.payment : 'pii_type' = 'financial_pci', 'pci_dss' = 'true'
silver.payment : 'pii_type' = 'financial_pci', 'pci_dss' = 'true'
silver.payment : 'pii_type' = 'address', 'gdpr_subject' = 'true'
silver.order : 'pii_type' = 'address', 'gdpr_subject' = 'true'


In [10]:
email = "user@domain.com"
mask = left(split(email, '@')[0], 1)
#concat(left(split(email, '@')[0], 1), '***@', split(email, '@')[1])
print(mask)

PySparkTypeError: [NOT_COLUMN_OR_STR] Argument `col` should be a Column or str, got int.

In [23]:
#phone = "+911234567890"
df = spark.createDataFrame( [("+911234567890",), ("+919876543210",)], ['phone'] )
df.withColumn('mask', concat(lit('****'), right(col('phone'), lit(4)))) \
    .select('phone', 'mask').show()

+-------------+--------+
|        phone|    mask|
+-------------+--------+
|+911234567890|****7890|
|+919876543210|****3210|
+-------------+--------+



----------------------------------------
Exception happened during processing of request from ('127.0.0.1', 63811)
Traceback (most recent call last):
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/socketserver.py", line 720, in __init__
    self.handle()
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/site-packages/pyspark/accumulators.py", line 267, in 

In [5]:
df_children = spark.createDataFrame(
  data = [("Mikhail", 15), ("Zaky", 13), ("Zoya", 8)],
  schema = ['name', 'age'])
#display(df_children)
df_children.show()

+-------+---+
|   name|age|
+-------+---+
|Mikhail| 15|
|   Zaky| 13|
|   Zoya|  8|
+-------+---+



In [6]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

df_children_with_schema = spark.createDataFrame(
  data = [("Mikhail", 15), ("Zaky", 13), ("Zoya", 8)],
  schema = StructType([
    StructField('name', StringType(), True),
    StructField('age', IntegerType(), True)
  ])
)
#display(df_children_with_schema)
df_children_with_schema.show()

+-------+---+
|   name|age|
+-------+---+
|Mikhail| 15|
|   Zaky| 13|
|   Zoya|  8|
+-------+---+



In [ ]:
# Assign this variable your full volume file path
volume_file_path = ""

df_csv = (spark.read
  .format("csv")
  .option("header", True)
  .option("inferSchema", True)
  .load(volume_file_path)
)
display(df_csv)

In [8]:
import requests

# Download data from URL
url = "https://api.fda.gov/drug/drugsfda.json?limit=100"
response = requests.get(url)

# Create the DataFrame
df_drugs = spark.createDataFrame(response.json()["results"])
#display(df_drugs)
df_drugs.show()

+------------------+--------------------+--------------------+--------------------+--------------------+
|application_number|            products|        sponsor_name|         submissions|             openfda|
+------------------+--------------------+--------------------+--------------------+--------------------+
|        ANDA074602|[{marketing_statu...|AUROBINDO PHARMA LTD|[{submission_stat...|                NULL|
|        ANDA074926|[{marketing_statu...|             COSETTE|[{submission_clas...|{spl_id -> [a19a2...|
|        ANDA075249|[{marketing_statu...|             BEDFORD|[{submission_stat...|                NULL|
|        ANDA070076|[{marketing_statu...|              RISING|[{submission_clas...|{product_ndc -> [...|
|        ANDA070397|[{marketing_statu...|         WATSON LABS|                NULL|                NULL|
|        ANDA070706|[{marketing_statu...|   ACTAVIS ELIZABETH|[{submission_clas...|                NULL|
|        ANDA071051|[{marketing_statu...|  FRESENIUS KA

In [12]:
#display(df_drugs.select(df_drugs["products"]))
df_drugs.select(df_drugs["products"]).show()

+--------------------+
|            products|
+--------------------+
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
|[{marketing_statu...|
+--------------------+
only showing top 20 rows



In [15]:
df_drugs.select(df_drugs["products"][0]["brand_name"]).show(truncate = False)

+-----------------------------+
|products[0][brand_name]      |
+-----------------------------+
|LACTULOSE                    |
|M-ZOLE 3 COMBINATION PACK    |
|MIDAZOLAM HYDROCHLORIDE      |
|METHYLDOPA                   |
|CLONIDINE HYDROCHLORIDE      |
|DIAZEPAM                     |
|ASTRAMORPH PF                |
|HALOPERIDOL                  |
|CLORAZEPATE DIPOTASSIUM      |
|CLONIDINE HYDROCHLORIDE      |
|HALOPERIDOL                  |
|POTASSIUM CHLORIDE           |
|INDAPAMIDE                   |
|PREDNISOLONE SODIUM PHOSPHATE|
|THIOTEPA                     |
|CALCITRIOL                   |
|TRILYTE                      |
|ZAROXOLYN                    |
|HALOPERIDOL                  |
|MEPERIDINE HYDROCHLORIDE     |
+-----------------------------+
only showing top 20 rows



----------------------------------------
Exception happened during processing of request from ('127.0.0.1', 64035)
Traceback (most recent call last):
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/socketserver.py", line 720, in __init__
    self.handle()
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/Users/abhinavmohanty/opt/anaconda3/lib/python3.8/site-packages/pyspark/accumulators.py", line 267, in 